# 05 · 主实验 Dense 基线：CASIA-WebFace → LFW 验证（M1）

| 模型 | 训练方式 | 作用 |
|---|---|---|
| **Dense-Clean** | ImageNet 预训练 ResNet-18 + 512 维嵌入（FC+BN）+ CosFace（s=64, m=0.35），只用干净图 | 参照：没见过遮挡的人脸模型 |
| **Dense-Aug** | 完全相同的初始权重、数据顺序、翻转、优化器和 epoch 数；唯一区别：每张训练图以 0.5 概率加随机方块遮挡（面积 10–30%） | **所有剪枝方法的共同起点** |

- 优化：SGD（momentum 0.9，lr 0.1，wd 5e-4），batch 256，1 epoch 线性 warm-up + cosine，20 epochs，bf16。
- 模型选择：留出 CASIA 身份的验证对上，干净 + 方块 10/30/50% 四种条件平均验证准确率最高的 epoch（两个模型同一规则）。
- 测试：LFW（主）、AgeDB-30、CALFW、CPLFW，各 6000 对，标准 10 折（9 折选阈值、1 折测试）准确率 mean ± std。

**运行时间**：每个模型约 45 分钟（RTX 3070 Ti Laptop）。每个 epoch 都保存 `checkpoints/<name>_last.pt`，中断后重新运行同一单元格会**自动断点续训**。
也可以在终端运行（不依赖 VS Code 窗口）：`python -m occprune.casia_pipeline train --mode clean`（PowerShell 中先执行 `$env:PYTHONPATH="src"`）。

## 0. 环境检查

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload：修改 src/occprune 下的代码后无需重启 kernel

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src" / "occprune").exists() else Path.cwd().parent  # 仓库根目录
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import os
os.chdir(ROOT)   # 相对路径（data/、results/）都以仓库根目录为准

import occprune
print("Python  :", sys.executable)
print("occprune:", occprune.__file__)

## 1. 设置

In [ ]:
import torch
from occprune.utils import load_config
from occprune.casia_pipeline import run_face_dense, summarize_face_dense

cfg = load_config("configs/casia.yaml")
SEED = cfg["seed"]
RETRAIN = False          # True = 重新训练（已有结果会被覆盖；有 _last.pt 时从断点继续）
# cfg["train"]["epochs"] = 20

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Dense-Clean

In [ ]:
res_clean = run_face_dense("clean", cfg, SEED, skip_if_exists=not RETRAIN)

## 3. Dense-Aug（剪枝起点）

In [ ]:
res_aug = run_face_dense("aug", cfg, SEED, skip_if_exists=not RETRAIN)

## 4. 结果表
LFW 各条件：10 折准确率 mean ± std（%）。`Sq` = 随机方块（与训练遮挡同分布）；`Eyes / Mouth+nose / Upper / Lower half` = 按区域遮挡（训练中**从未出现**）。
表格同时保存为 `results/casia/casia_dense_table_seed42.csv/.md`，图保存在 `figures/`。

In [ ]:
table, figs = summarize_face_dense(SEED)
table.set_index("Method").T

In [ ]:
from IPython.display import Image, display
display(Image("figures/casia_dense_lfw_seed42.png", width=900))
display(Image("figures/casia_dense_training_curves_seed42.png", width=900))

## 5. 差异是否显著：配对 bootstrap
两个模型在同一批 6000 对上评估：各自用 10 折阈值得到每一对的对错，再对**对**有放回重采样 2000 次，得到 Dense-Aug − Dense-Clean 准确率差的 95% 置信区间。区间跨 0 时不下结论。

In [ ]:
import pandas as pd
pd.DataFrame(figs["bootstrap"]).T.mul(100).round(2)

## 6. 梯度诊断（为剪枝打分做准备）
固定 256 张训练图（BN eval 模式），干净 vs 遮挡下的 CosFace loss 与卷积权重梯度范数。开发集上遮挡梯度约为干净的 20 倍；CASIA 训练不会完全拟合，这一比值需要重新测量（计划第 8 节）。

In [ ]:
pd.DataFrame({m: r["train_diagnostics"] for m, r in [("Dense-Clean", res_clean), ("Dense-Aug", res_aug)]})